In [ ]:
{
 "cells": [
  {
   "cell_type": "code",
   "execution_count": 1,
   "id": "e6d51f8f-ed9c-4840-9792-54f6c595448c",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "C:\\Users\\shravya\\Desktop\\Network Security Project\\NetworkSecurity\n"
     ]
    }
   ],
   "source": [
    "import os\n",
    "print(os.getcwd())"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 3,
   "id": "a354337a-409a-46a8-9922-6291626960ee",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Ratings DataFrame:\n",
      "   user_id  movie_id  rating  timestamp\n",
      "0      196       242       3  881250949\n",
      "1      186       302       3  891717742\n",
      "2       22       377       1  878887116\n",
      "3      244        51       2  880606923\n",
      "4      166       346       1  886397596\n",
      "\n",
      "Movies DataFrame:\n",
      "   movie_id              title\n",
      "0         1   Toy Story (1995)\n",
      "1         2   GoldenEye (1995)\n",
      "2         3  Four Rooms (1995)\n",
      "3         4  Get Shorty (1995)\n",
      "4         5     Copycat (1995)\n"
     ]
    }
   ],
   "source": [
    "import pandas as pd\n",
    "from surprise import Dataset, Reader, SVD\n",
    "from surprise.model_selection import train_test_split\n",
    "from surprise import accuracy\n",
    "\n",
    "# Load ratings and movie titles\n",
    "ratings = pd.read_csv('ml-100k/u.data' if os.path.exists('ml-100k/u.data') else 'u.data', sep='\\t', names=['user_id', 'movie_id', 'rating', 'timestamp'])\n",
    "movies = pd.read_csv('ml-100k/u.item' if os.path.exists('ml-100k/u.item') else 'u.item', sep='|', encoding='latin-1', \n",
    "                     names=['movie_id', 'title', 'release_date', 'video_release', 'url'] + [f'genre_{i}' for i in range(19)])\n",
    "\n",
    "# Prepare data for Surprise\n",
    "reader = Reader(rating_scale=(1, 5))\n",
    "data = Dataset.load_from_df(ratings[['user_id', 'movie_id', 'rating']], reader)\n",
    "\n",
    "# Verify data\n",
    "print(\"Ratings DataFrame:\")\n",
    "print(ratings.head())\n",
    "print(\"\\nMovies DataFrame:\")\n",
    "print(movies[['movie_id', 'title']].head())"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 5,
   "id": "23c2cad0-cb31-4a4c-992b-9378c784ed73",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "RMSE: 0.9352\n",
      "RMSE: 0.935171451026933\n"
     ]
    }
   ],
   "source": [
    "# Split data into train and test sets\n",
    "trainset, testset = train_test_split(data, test_size=0.2, random_state=42)\n",
    "\n",
    "# Train SVD model\n",
    "model = SVD(n_factors=100, random_state=42)\n",
    "model.fit(trainset)\n",
    "\n",
    "# Evaluate model\n",
    "predictions = model.test(testset)\n",
    "rmse = accuracy.rmse(predictions)\n",
    "print(f\"RMSE: {rmse}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 7,
   "id": "ad3bd983-6e39-406f-a296-721ccac947de",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Top 10 recommendations for user 1:\n",
      "Movie: Star Wars (1977), Predicted Rating: 5.00\n",
      "Movie: Empire Strikes Back, The (1980), Predicted Rating: 4.97\n",
      "Movie: Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1963), Predicted Rating: 4.94\n",
      "Movie: One Flew Over the Cuckoo's Nest (1975), Predicted Rating: 4.94\n",
      "Movie: Rear Window (1954), Predicted Rating: 4.93\n",
      "Movie: Hoop Dreams (1994), Predicted Rating: 4.92\n",
      "Movie: Citizen Kane (1941), Predicted Rating: 4.75\n",
      "Movie: Wrong Trousers, The (1993), Predicted Rating: 4.73\n",
      "Movie: Wallace & Gromit: The Best of Aardman Animation (1996), Predicted Rating: 4.72\n",
      "Movie: 12 Angry Men (1957), Predicted Rating: 4.71\n"
     ]
    }
   ],
   "source": [
    "def get_top_n_recommendations(user_id, n=10):\n",
    "    all_movie_ids = ratings['movie_id'].unique()\n",
    "    predictions = [model.predict(user_id, movie_id) for movie_id in all_movie_ids]\n",
    "    predictions.sort(key=lambda x: x.est, reverse=True)\n",
    "    top_n = [(pred.iid, pred.est) for pred in predictions[:n]]\n",
    "    return top_n\n",
    "\n",
    "# Test recommendations for user ID 1\n",
    "user_id = 1\n",
    "recommendations = get_top_n_recommendations(user_id)\n",
    "print(f\"Top 10 recommendations for user {user_id}:\")\n",
    "for movie_id, predicted_rating in recommendations:\n",
    "    title = movies[movies['movie_id'] == movie_id]['title'].values[0]\n",
    "    print(f\"Movie: {title}, Predicted Rating: {predicted_rating:.2f}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 9,
   "id": "cede67e0-ea41-4181-8b0e-879ee323a230",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Create a dictionary of movie IDs to titles for JavaScript\n",
    "movie_titles = movies.set_index('movie_id')['title'].to_dict()\n",
    "\n",
    "# Function to get recommendations as a list of dictionaries\n",
    "def get_recommendations_for_frontend(user_id):\n",
    "    recommendations = get_top_n_recommendations(user_id)\n",
    "    return [{'movie_id': int(movie_id), 'title': movie_titles[movie_id], 'predicted_rating': round(pred_rating, 2)} \n",
    "            for movie_id, pred_rating in recommendations]"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 1,
   "id": "2990b730-0502-45b3-b862-78696c165b2d",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "application/vnd.jupyter.widget-view+json": {
       "model_id": "e4faed090a5c43da84845e2f669e7f73",
       "version_major": 2,
       "version_minor": 0
      },
      "text/plain": [
       "Dropdown(description='User ID:', options=(1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 2…"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    },
    {
     "data": {
      "application/vnd.jupyter.widget-view+json": {
       "model_id": "bc39170fed3846609a5c34f0ebb9e360",
       "version_major": 2,
       "version_minor": 0
      },
      "text/plain": [
       "Output()"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "import ipywidgets as widgets\n",
    "from IPython.display import display\n",
    "\n",
    "# Create dropdown for user IDs\n",
    "user_ids = list(range(1, 944))\n",
    "dropdown = widgets.Dropdown(options=user_ids, description='User ID:', style={'description_width': 'initial'})\n",
    "\n",
    "# Create output area\n",
    "output = widgets.Output()\n",
    "\n",
    "# Function to display recommendations\n",
    "def show_recommendations(change):\n",
    "    with output:\n",
    "        output.clear_output()\n",
    "        user_id = change['new']\n",
    "        recommendations = get_top_n_recommendations(user_id)\n",
    "        print(f\"Top 10 recommendations for user {user_id}:\")\n",
    "        for movie_id, predicted_rating in recommendations:\n",
    "            title = movies[movies['movie_id'] == movie_id]['title'].values[0]\n",
    "            print(f\"Movie: {title}, Predicted Rating: {predicted_rating:.2f}\")\n",
    "\n",
    "# Link dropdown to function\n",
    "dropdown.observe(show_recommendations, names='value')\n",
    "\n",
    "# Display the interface\n",
    "display(dropdown)\n",
    "display(output)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "223b9010-9ae7-42c1-9f64-b74301e1e64a",
   "metadata": {},
   "outputs": [],
   "source": []
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3 (ipykernel)",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.12.3"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}